In [3]:
import os
import time
from typing import List
from typing import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph import END

In [2]:
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import Tool
from langchain.agents import initialize_agent
from langchain.agents import AgentType

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

In [5]:
# Define state type
class PlannerExecutorState(TypedDict):
    input: str
    steps: List[str]
    output: str

In [6]:
# Gemini LLM setup
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-pro",
    temperature=0.7,
    convert_system_message_to_human=True
)

In [7]:
# Tool setup
tavily_tool = TavilySearchResults()
tools = [
    Tool(
        name="WebSearch",
        func=tavily_tool.run,
        description="Useful for answering questions from the web"
    )
]

In [8]:
# Planner Node (Structured Output)
def planner_node(state: PlannerExecutorState) -> PlannerExecutorState:
    question = state["input"]
    prompt = (
        "You are a task planner. Decompose the user request into numbered tasks only.\n"
        f"User Request: {question}\n"
        "Respond only as:\n"
        "1. Task one...\n2. Task two...\nDo not include any introduction or conclusion."
    )
    try:
        response = llm.invoke(prompt)
    except Exception as e:
        print("Gemini Error:", e)
        raise

    print("Gemini Planner Output:\n", response.content)

    # Only extract numbered steps
    lines = response.content.split("\n")
    steps = [
        line.strip()
        for line in lines
        if line.strip() and line.strip()[0].isdigit()
    ]
    state["steps"] = steps
    return state

In [9]:
# Executor Node
def executor_node(state: PlannerExecutorState) -> PlannerExecutorState:
    steps = state.get("steps", [])
    executor = initialize_agent(
        tools, llm, agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, verbose=False
    )

    results = []
    for step in steps:
        try:
            print(f"\nExecuting Step: {step}")
            result = executor.run(step)
            print("Result:", result)
            results.append(f"Step: {step}\n{result}")
            time.sleep(2)  # Add delay to avoid rate limit
        except Exception as e:
            print("Step failed:", e)
            results.append(f"Step: {step}\nError: {str(e)}")

    state["output"] = "\n\n".join(results)
    return state

In [10]:
# LangGraph Build
builder = StateGraph(PlannerExecutorState)
builder.add_node("planner", planner_node)
builder.add_node("executor", executor_node)

builder.set_entry_point("planner")
builder.add_edge("planner", "executor")
builder.add_edge("executor", END)

In [11]:
graph = builder.compile()

In [12]:
# Run the graph
query = "Tell me about Elon Musk"
input_state = {"input": query, "steps": [], "output": ""}
final_state = graph.invoke(input_state)

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Gemini Planner Output:
 1.  Gather information on his early life, education, and first companies (Zip2, X.com/PayPal).
2.  List the major companies he founded or leads, including SpaceX and Tesla.
3.  Detail the key innovations and goals of his primary companies, such as reusable rockets and the mass-market adoption of electric vehicles.
4.  Describe his other significant ventures, such as Neuralink, The Boring Company, and xAI.
5.  Explain his acquisition and rebranding of Twitter to X.
6.  Summarize his long-term vision for humanity, focusing on space colonization and sustainable energy.

Executing Step: 1.  Gather information on his early life, education, and first companies (Zip2, X.com/PayPal).


C:\Users\emon1\AppData\Local\Temp\ipykernel_4100\3955840667.py:4: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  executor = initialize_agent(
C:\Users\emon1\AppData\Local\Temp\ipykernel_4100\3955840667.py:12: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = executor.run(step)
c:\Users\emon1\AppData\

Step failed: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Here is a summary of Elon Musk's early life, education, and his first companies:

### Early Life

Elon Reeve Musk was born on June 28, 1971, in Pretoria, South Africa. His mother, Maye Musk, is a Canadian model and dietitian, and his father, Errol Musk, is a wealthy South African engineer. Musk spent his early childhood in South Africa with his brother Kimbal and sister Tosca.

From a young age, Musk showed a keen interest in computers and technology. He taught himself how to program, and at the age of 12, he developed and sold his first video game, a space-themed game called *Blastar*, for about $500.

In 1988, at the age of 17, Musk moved to Canada to attend university and to avoid the mandatory military service in South Africa.

### Education

Musk began his higher e

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


...

Step failed: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Here is a description of Elon Musk's other significant ventures: Neuralink, The Boring Company, and xAI.

**Neuralink** is a neurotechnology company aiming to create a high-bandwidth, implantable brain-computer interface (BCI). The company's short-term goal is to develop devices to treat serious brain diseases and help people with paralysis regain the ability to communicate and interact with technology by controlling computers and mobile devices with their thoughts. In early 2024, Neuralink successfully implanted its first device in a human patient. The long-term vision for Neuralink is far more ambitious, with Elon Musk speaking of a future where such technology could lead to a symbiotic relationship between humans and artificial intelligence, a concept often referred 

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Step failed: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Elon Musk's acquisition of Twitter was a multi-billion dollar saga that unfolded over several months in 2022, culminating in a dramatic rebranding of the social media platform to "X" in 2023. Here's an explanation of both the acquisition and the rebranding:

### The Acquisition of Twitter

The acquisition process was marked by a series of twists and turns:

1.  **Building a Stake:** In early 2022, Elon Musk began quietly purchasing shares of Twitter. By April, he had become the company's largest shareholder with a 9.2% stake.

2.  **The Board Seat Offer:** Twitter's management, including then-CEO Parag Agrawal, offered Musk a seat on the board of directors. Musk initially accepted but then reversed his decision, a move that signaled he had bigger plans than simply being

c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")
c:\Users\emon1\AppData\Local\Programs\Python\Python311\Lib\site-packages\langchain_google_genai\chat_models.py:353: UserWarning: Convert_system_message_to_human will be deprecated!
  warnings.warn("Convert_system_message_to_human will be deprecated!")


Step failed: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Elon Musk's long-term vision for humanity is a two-pronged approach aimed at ensuring the long-term survival and prosperity of the human species.

1.  **Sustainable Energy:** Musk's primary goal on Earth is to accelerate the world's transition to sustainable energy. He believes humanity's reliance on fossil fuels is unsustainable and a significant threat to our future. His strategy involves a vertically integrated system:
    *   **Energy Generation:** Through Tesla Solar, he aims to make solar power a primary source of clean energy generation for homes and businesses.
    *   **Energy Storage:** With Tesla Energy products like the Powerwall and Megapack, he provides solutions to store this renewable energy, ensuring a consistent power supply even when the sun isn't shi

In [14]:
print("\nFinal Answer:\n", final_state["output"])


Final Answer:
 Step: 1.  Gather information on his early life, education, and first companies (Zip2, X.com/PayPal).
Error: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Here is a summary of Elon Musk's early life, education, and his first companies:

### Early Life

Elon Reeve Musk was born on June 28, 1971, in Pretoria, South Africa. His mother, Maye Musk, is a Canadian model and dietitian, and his father, Errol Musk, is a wealthy South African engineer. Musk spent his early childhood in South Africa with his brother Kimbal and sister Tosca.

From a young age, Musk showed a keen interest in computers and technology. He taught himself how to program, and at the age of 12, he developed and sold his first video game, a space-themed game called *Blastar*, for about $500.

In 1988, at the age of 17, Musk moved to Canada to attend 